In [ ]:
from __future__ import annotations

PS-P2VP bragg onion in air environment.

Investigate
- wavelength dependent forward to backward scattering ratio
- target wavelength and number of layers dependent colour performance
- angular distribution of scattered light
- Poynting streamlines

therefor, vary
- target wavelength
- number of layers

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
import pandas as pd


from bragg_onion.materials import MaterialFileSpec, ConstantDispersion, load_materials
from bragg_onion.sweep import (
    run_bragg_onion_sweep,
    make_plotting_aliases,
    plot_sweep_metric,
    plot_sweep_heatmap,
    plot_sweep_colour_strip,
)

from bragg_onion.scattering_analysis import (
    plot_differential_scattering_vs_angle,
    plot_polar_scattering,
    plot_forward_side_backward_vs_wavelength,
    plot_partition_pie_at_wavelength,
    print_scattering_metrics,
    scattering_metrics_dataframe,
)

from bragg_onion.colour_solid_plotting import (
    load_rosch_macadam_colour_solid,
    plot_rosch_macadam_colour_solid,
    add_cielab_point,
)

import bragg_onion.fields as bo_fields

plt.rcParams["figure.dpi"] = 600


In [ ]:
# Import refractive indices
FILE_PS = r"C:\Users\SchwarzN\OneDrive - Université de Fribourg\Institution\P1_BraggSpericalPigments\Simulation\RefractiveIndices\PS(hPS)\x=0.0.txt"
FILE_P2VP = r"C:\Users\SchwarzN\OneDrive - Université de Fribourg\Institution\P1_BraggSpericalPigments\Simulation\RefractiveIndices\P2VP(PDP)\x=0.0.txt"

# Sweep settings
WAVELENGTHS_NM = np.arange(350, 830, 1)
THETA_DEG = np.linspace(0, 180, 361)

COLLECTION_NA = 0.8
COLLECTION_DIRECTION = "backward"
INTEGRATION_QUANTITY_FOR_COLOUR = "q_collected_geom"

PARAMETER_GRID = {
    "peak_wavelength_m": np.arange(350, 830, 50) * 1e-9,
    "n_layers": [5, 20, 50],
    "outer_layer": ["A"],
    "core_thickness_factor": [0.5],
}

# Additional geometry kwargs applied to every case.
# Keep empty {} if not needed.
FIXED_GEOMETRY_KWARGS: dict = {}

# ============================================================
# Plotting settings
# ============================================================
def apply_one_decimal_axis_format(ax) -> None:
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))


def apply_one_decimal_xaxis(ax) -> None:
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))


def apply_one_decimal_yaxis(ax) -> None:
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))

# ============================================================
# Output settings
# ============================================================
root_candidate = Path.cwd().resolve()
ROOT = root_candidate.parent if root_candidate.name.lower() == "notebooks" else root_candidate
OUTPUT_DIR = ROOT / "outputs" / "analyse_PS_P2VP"


# ============================================================
# Helpers
# ============================================================
def ensure_output_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def save_show_close_figure(
    path: Path | None = None,
    *,
    fig=None,
    dpi: int = 600,
    show: bool = True,
    close: bool = False,
    tight: bool = True,
) -> None:
    """
    Save a Matplotlib figure, show it in the notebook, and optionally close it.

    Parameters
    ----------
    path :
        Output file path. If None, the figure is not saved.
    fig :
        Matplotlib figure object. If None, uses plt.gcf().
    dpi :
        Save resolution.
    show :
        If True, display the figure in the notebook.
    close :
        If True, close the figure after saving/showing.
    tight :
        If True, apply tight_layout() before saving/showing.
    """
    if fig is None:
        fig = plt.gcf()

    if tight:
        fig.tight_layout()

    if path is not None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        fig.savefig(path.with_suffix(".pdf"), bbox_inches="tight")

    if show:
        plt.show()

    if close:
        plt.close(fig)

# Define grids
wavelengths_m = WAVELENGTHS_NM * 1e-9
theta_rad = np.deg2rad(THETA_DEG)

In [ ]:
# Load materials
material_specs = {
    "PS": MaterialFileSpec(
        name="PS",
        path=FILE_PS,
        wavelength_unit="nm",
        skiprows=2,
        names=["Wavelength", "RefractiveIndex", "k"],
        n_column="RefractiveIndex",
        k_column="k",
        extrapolation="extrapolate",
    ),
    "P2VP": MaterialFileSpec(
        name="P2VP",
        path=FILE_P2VP,
        wavelength_unit="nm",
        skiprows=2,
        names=["Wavelength", "RefractiveIndex", "k"],
        n_column="RefractiveIndex",
        k_column="k",
        extrapolation="extrapolate",
    ),

}

print("Loading materials...")
loaded = load_materials(material_specs)

mat_A = loaded["PS"]
mat_B = loaded["P2VP"]

# Constant air medium as dispersion object for the surrounding medium.
medium = ConstantDispersion.from_nk("air", n=1.0, k=0.0)
# medium = 1.0

print("... materials loaded")

In [ ]:
# Run the sweep
print("Running Bragg onion sweep...")
sweep_result = run_bragg_onion_sweep(
    geometry_mode = "peak_wavelength",
    parameter_grid=PARAMETER_GRID,
    material_a=mat_A,
    material_b=mat_B,
    n_medium=medium,
    wavelengths_m=wavelengths_m,
    theta_rad=theta_rad,
    collection_na=COLLECTION_NA,
    collection_direction=COLLECTION_DIRECTION,
    integration_quantity_for_colour=INTEGRATION_QUANTITY_FOR_COLOUR,
    colour_wavelength_min_nm=None,
    colour_wavelength_max_nm=None,
    colour_normalize_input=False,
    colour_normalization="max", # irrelevant when colour_normalize_input=False
    fixed_geometry_kwargs=FIXED_GEOMETRY_KWARGS,
    store_full_results=True,
    progress=True,
)

In [ ]:
# Generate Figures
print("Generating figures...")
# All cases in Rosch-Macadam colour solid
# Raw and alias-renamed sweep tables
df_raw = sweep_result.dataframe
df_plot = make_plotting_aliases(df_raw)

print(df_plot.columns.tolist())
print(df_plot.head)

In [ ]:

# 1) All sweep cases inside the Rosch–MacAdam colour solid
solid_df = load_rosch_macadam_colour_solid()

fig = plot_rosch_macadam_colour_solid(
    solid_df,
    title="Sweep results in Rosch–MacAdam colour solid",
    max_points=50000,
    marker_size=1.2,
    opacity=0.06,
)

for _, row in df_plot.iterrows():
    extra_hover = [
        f"design λ: {row['design_peak_wavelength_nm']:.1f} nm",
        f"n_layers: {int(row['n_layers'])}",
        f"outer_layer: {row['outer_layer']}",
        f"eta_C: {row['eta_C']:.3f}",
        f"eta_L: {row['eta_L']:.3f}",
        f"eta_Y: {row['eta_Y']:.3f}",
        f"hex: {row['hex']}",
    ]

    fig = add_cielab_point(
        fig,
        L=float(row["L"]),
        a=float(row["a"]),
        b=float(row["b"]),
        name=f"case {int(row['case_index'])}",
        color=str(row["hex"]),
        size=6.5 + 5.0 * float(row["eta_C"]),   # marker size scaled by eta_C
        symbol="diamond",
        line_color="black",
        line_width=1.0,
        extra_hover_lines=extra_hover,
    )

fig.update_layout(
    scene=dict(
        xaxis_title="a*",
        yaxis_title="b*",
        zaxis_title="L*",
        xaxis=dict(title="a*", tickformat=".1f"),
        yaxis=dict(title="b*", tickformat=".1f"),
        zaxis=dict(title="L*", tickformat=".1f"),
        aspectmode="cube",
        camera=dict(eye=dict(x=1.8, y=1.8, z=1.2)),
    )
)


html_path = OUTPUT_DIR / "rosch_macadam_colour_solid.html"
fig.write_html(str(html_path))

fig.show()
print(f"Saved interactive 3D figure to: {html_path}")

In [ ]:
# 2) eta_C heatmap with n_layers and design_peak_wavelength_nm on the in-plane axes
for outer_layer in sorted(df_plot["outer_layer"].unique()):
    subset = df_plot[df_plot["outer_layer"] == outer_layer]

    plot_sweep_heatmap(
        subset,
        x="design_peak_wavelength_nm",
        y="n_layers",
        value="eta_C",
        scale="linear",
    )

    plt.title(f"eta_C heatmap (outer layer = {outer_layer})")
    save_show_close_figure(
        OUTPUT_DIR / f"eta_C_heatmap_outer_{outer_layer}.png",
        dpi=600,
        show=True,
        close=False,
    )


In [ ]:
# 3) Spectra for all sweep cases
# A — plot backward-collected spectra for all sweep cases
if len(sweep_result.case_results) == 0:
    raise RuntimeError(
        "No full case results are stored. Rerun the sweep with store_full_results=True."
    )

fig, ax = plt.subplots(figsize=(8, 5))

for case in sweep_result.case_results:
    wl_nm = case.integrated_result.wavelengths_m * 1e9
    spec = case.integrated_result.q_collected_geom

    # Use the final plotted colour if available
    row = df_plot.loc[df_plot["case_index"] == case.case_index].iloc[0]
    color = row["hex"]

    label = (
        f"λd={row['design_peak_wavelength_nm']:.0f} nm, "
        f"n={int(row['n_layers'])}, "
        f"outer={row['outer_layer']}"
    )

    ax.plot(wl_nm, spec, color=color, linewidth=1.5, label=label)

ax.set_title("Backward-collected spectra")
ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("Backward collected efficiency [-]")

# B — Qsca spectra instead
fig, ax = plt.subplots(figsize=(8, 5))

for case in sweep_result.case_results:
    wl_nm = case.scattering_result.wavelengths_m * 1e9
    spec = case.scattering_result.qsca

    row = df_plot.loc[df_plot["case_index"] == case.case_index].iloc[0]
    color = row["hex"]

    label = (
        f"λd={row['design_peak_wavelength_nm']:.0f} nm, "
        f"n={int(row['n_layers'])}, "
        f"outer={row['outer_layer']}"
    )

    ax.plot(wl_nm, spec, color=color, linewidth=1.5, label=label)

ax.set_title("Scattering efficiency spectra")
ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("Scattering efficiency Qsca")

In [ ]:
# 4) Max backward-collected signal vs design wavelength
fig, ax = plt.subplots(figsize=(7, 4.5))

for (n_layers, outer_layer), grp in df_plot.groupby(["n_layers", "outer_layer"]):
    grp = grp.sort_values("design_peak_wavelength_nm")

    ax.plot(
        grp["design_peak_wavelength_nm"],
        grp["Ccollected_max_m2"],
        marker="o",
        linewidth=1.5,
        label=f"n_layers={n_layers}, outer={outer_layer}",
    )

ax.set_xlabel("Design peak wavelength [nm]")
ax.set_ylabel("Max backward-collected signal [m²]")
ax.set_title("Max backward-collected signal vs design wavelength")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

# Same but in dB scale
fig, ax = plt.subplots(figsize=(7, 4.5))

for (n_layers, outer_layer), grp in df_plot.groupby(["n_layers", "outer_layer"]):
    grp = grp.sort_values("design_peak_wavelength_nm")
    y_DB = 10 * np.log10(grp["Ccollected_max_m2"])

    ax.plot(
        grp["design_peak_wavelength_nm"],
        y_DB,
        marker="o",
        linewidth=1.5,
        label=f"n_layers={n_layers}, outer={outer_layer}",
    )

ax.set_xlabel("Design peak wavelength [nm]")
ax.set_ylabel("Max backward-collected signal [dB]")
ax.set_title("Max backward-collected signal vs design wavelength (dB)")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# 5) Computed colour strip
for outer_layer in sorted(df_plot["outer_layer"].unique()):
    subset = df_plot[df_plot["outer_layer"] == outer_layer].sort_values(
        by=["design_peak_wavelength_nm", "n_layers"]
    )

    plot_sweep_colour_strip(
        subset,
        x="design_peak_wavelength_nm",
        colour_hex_col="hex",
        label_col="n_layers",
    )
    plt.title(f"Computed colour strip (outer layer = {outer_layer}, labels = n_layers)")
    plt.show()

# Combined colour strip for all cases
plot_sweep_colour_strip(
    df_plot.sort_values(by=["outer_layer", "design_peak_wavelength_nm", "n_layers"]),
    x="design_peak_wavelength_nm",
    colour_hex_col="hex",
    label_col="n_layers",
)
plt.title("Computed colour strip (labels = n_layers)")
plt.show()

In [ ]:
# 6) All sweep cases in the CIELAB a*–b* plane
fig, ax = plt.subplots(figsize=(6, 6))

for _, row in df_plot.iterrows():
    ax.scatter(
        row["a"],
        row["b"],
        color=row["hex"],
        s=120,
        edgecolor="black",
        linewidth=0.7,
    )

    # Optional small text annotation:
    ax.text(
        row["a"] + 0.8,
        row["b"] + 0.8,
        f"{int(row['n_layers'])}/{row['outer_layer']}",
        fontsize=7,
    )

ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
ax.set_aspect("equal")
ax.set_xlabel("a*")
ax.set_ylabel("b*")
ax.set_title("Sweep results in CIELAB a*–b* plane")
plt.tight_layout()
plt.show()

In [ ]:
# Helper to select single case from sweep
def select_sweep_case(
    df_plot: pd.DataFrame,
    sweep_result,
    *,
    design_peak_wavelength_nm: float | None = None,
    n_layers: int | None = None,
    outer_layer: str | None = None,
    case_index: int | None = None,
):
    """
    Return (row, case_result) for one selected sweep case.
    """
    if case_index is not None:
        row = df_plot.loc[df_plot["case_index"] == case_index].iloc[0]
    else:
        mask = np.ones(len(df_plot), dtype=bool)

        if design_peak_wavelength_nm is not None:
            mask &= np.isclose(
                df_plot["design_peak_wavelength_nm"].to_numpy(dtype=float),
                float(design_peak_wavelength_nm),
                atol=1e-6,
            )

        if n_layers is not None:
            mask &= df_plot["n_layers"].to_numpy(dtype=int) == int(n_layers)

        if outer_layer is not None:
            mask &= df_plot["outer_layer"].astype(str).to_numpy() == str(outer_layer)

        subset = df_plot.loc[mask]
        if subset.empty:
            raise ValueError("No sweep case matched the requested filters.")
        if len(subset) > 1:
            print("Multiple cases matched; taking the first one.")
        row = subset.iloc[0]

    case_idx = int(row["case_index"])

    matches = [c for c in sweep_result.case_results if c.case_index == case_idx]
    if not matches:
        raise RuntimeError(
            "No matching full case result found. "
            "Make sure the sweep was run with store_full_results=True."
        )

    return row, matches[0]

In [ ]:
# Choose one case for detailed analysis
row_sel, case_sel = select_sweep_case(
    df_plot,
    sweep_result,
    design_peak_wavelength_nm=550.0,
    n_layers=20,
    outer_layer="A",
)

print("Selected case:")
display(row_sel.to_frame().T)

In [ ]:
# Quick access objects
scat = case_sel.scattering_result
integ = case_sel.integrated_result
colour = case_sel.colour_result

design_wavelength_nm = float(row_sel["design_peak_wavelength_nm"])
print("Design wavelength ", design_wavelength_nm)

1. Metrics

In [ ]:
# Metrics summary (g, albedo, partitions, efficiencies, etc.)

# polar angle for dividing the solid angle into thirds
sphere_area_fract = 1/3
alpha_deg= np.rad2deg(np.arccos(1-4*np.pi * sphere_area_fract / (2*np.pi)))

print_scattering_metrics(
    scat,
    design_wavelength_nm=design_wavelength_nm,
    alpha_deg=alpha_deg,   # choose your forward/backward cone half-angle here
)

# as DataFrame
metrics_df = scattering_metrics_dataframe(
    scat,
    design_wavelength_nm=design_wavelength_nm,
    alpha_deg=alpha_deg,
)
display(metrics_df)

In [ ]:
# Plot average scattering direction g(λ) and albedo(λ) (Q_sca/Q_ext)
# g > 0 → forward-biased scattering
# g < 0 → backward-biased scattering
# g ≈ 0 → roughly symmetric redistribution
# albedo ≈ 1 → mostly scattering
# albedo ≈ 0 → mostly absorbing
wl_nm = scat.wavelengths_m * 1e9

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(wl_nm, scat.g, label="g (asymmetry parameter)", linewidth=1.8)
ax.axhline(0.0, color="gray", linestyle="--", linewidth=1.0)
ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("g")
ax.set_title("Asymmetry parameter vs wavelength")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(wl_nm, scat.albedo, label="Albedo = Qsca / Qext", linewidth=1.8)
ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("Albedo")
ax.set_ylim(-0.02, 1.02)
ax.set_title("Albedo vs wavelength")
ax.legend()
plt.tight_layout()
plt.show()

2. Angular scattering

In [ ]:
# Differential scattering vs angle for all wavelengths
# A) Geometric-cross-section normalized differential scattering

plot_differential_scattering_vs_angle(
    scat,
    wavelengths_nm=None,       # all wavelengths
    normalization="geom",      # (dσ/dΩ)/(πR²)
    as_db=False,
    cmap="turbo",
    alpha=0.6,
    linewidth=1.0,
)
plt.title("Differential scattering vs angle (geom-normalized, all wavelengths)")
plt.show()

# B) Same in dB
plot_differential_scattering_vs_angle(
    scat,
    wavelengths_nm=None,
    normalization="geom",
    as_db=True,
    cmap="turbo",
    alpha=0.6,
    linewidth=1.0,
)
plt.title("Differential scattering vs angle (geom-normalized, dB)")
plt.show()

# C) Phase function
plot_differential_scattering_vs_angle(
    scat,
    wavelengths_nm=None,
    normalization="sca",       # phase function
    as_db=False,
    cmap="viridis",
    alpha=0.6,
    linewidth=1.0,
)
plt.title("Phase function vs angle (all wavelengths)")
plt.show()

In [ ]:
# Differential scattering vs angle for selected wavelengths only
plot_differential_scattering_vs_angle(
    scat,
    wavelengths_nm=[450, 500, 550, 600, 650],
    normalization="geom",
    as_db=False,
    cmap="plasma",
    alpha=0.9,
    linewidth=1.8,
)
plt.title("Differential scattering vs angle (selected wavelengths)")
plt.show()

In [ ]:
# Polar plots
# A) Single wavelength (design wavelength)
plot_polar_scattering(
    scat,
    wavelengths_nm=design_wavelength_nm,
    normalization="geom",
    as_db=False,
    alpha=0.95,
    mirror=False,
)
plt.show()

# B) Multiple wavelengths
plot_polar_scattering(
    scat,
    wavelengths_nm=[450, 500, 550, 600, 650],
    normalization="geom",
    as_db=False,
    alpha=0.65,
    mirror=False,
)
plt.show()

# C) Multiple wavelengths in dB
plot_polar_scattering(
    scat,
    wavelengths_nm=[450, 500, 550, 600, 650],
    normalization="geom",
    as_db=True,
    alpha=0.65,
    mirror=False,
)
plt.show()

3. Partitioning

In [ ]:
# forward / side / backward spectra vs wavelength
ALPHA_DEG = 30.0
# polar angle for dividing the solid angle into thirds
sphere_area_fract = 1/3
ALPHA_DEG= np.rad2deg(np.arccos(1-4*np.pi * sphere_area_fract / (2*np.pi)))

In [ ]:
# A) Absolute integrated cross-sections
plot_forward_side_backward_vs_wavelength(
    scat,
    alpha_deg=ALPHA_DEG,
    normalize="none",
    as_db=False,
)
plt.title(f"Forward / side / backward integrated cross-sections (α = {ALPHA_DEG:.1f}°)")
plt.show()

In [ ]:
# B) Geometric-cross-section normalized efficiencies
plot_forward_side_backward_vs_wavelength(
    scat,
    alpha_deg=ALPHA_DEG,
    normalize="geom",
    as_db=False,
)
plt.title(f"Forward / side / backward integrated efficiencies (α = {ALPHA_DEG:.1f}°)")
plt.show()

In [ ]:
# C) Fractions of total scattering
plot_forward_side_backward_vs_wavelength(
    scat,
    alpha_deg=ALPHA_DEG,
    normalize="sca",
    as_db=False,
)
plt.title(f"Forward / side / backward fractions of total scattering (α = {ALPHA_DEG:.1f}°)")
plt.show()

In [ ]:
# D) Same in dB
plot_forward_side_backward_vs_wavelength(
    scat,
    alpha_deg=ALPHA_DEG,
    normalize="geom",
    as_db=True,
)
plt.title(f"Forward / side / backward integrated efficiencies (dB, α = {ALPHA_DEG:.1f}°)")
plt.show()

In [ ]:
# Pie chart of forward / side / backward / absorption
# A) At design wavelength
plot_partition_pie_at_wavelength(
    scat,
    alpha_deg=ALPHA_DEG,
    design_wavelength_nm=design_wavelength_nm,
    include_absorption=True,
)
plt.show()

# B) At an explicitly chosen wavelength
plot_partition_pie_at_wavelength(
    scat,
    alpha_deg=ALPHA_DEG,
    wavelength_nm=600.0,
    include_absorption=True,
)
plt.show()

4. Colour-driving spectrum vs. standard quantities

In [ ]:
# Compare selected case colour-driving spectrum to scattering quantities
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.plot(
    integ.wavelengths_m * 1e9,
    integ.q_collected_geom,
    label="q_collected_geom (colour-driving quantity)",
    linewidth=2.0,
)

ax.plot(
    scat.wavelengths_m * 1e9,
    scat.qsca,
    label="Qsca",
    linewidth=1.5,
    alpha=0.8,
)

ax.plot(
    scat.wavelengths_m * 1e9,
    scat.qext,
    label="Qext",
    linewidth=1.5,
    alpha=0.8,
)

ax.plot(
    scat.wavelengths_m * 1e9,
    scat.qabs,
    label="Qabs",
    linewidth=1.5,
    alpha=0.8,
)

ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("Dimensionless quantity [-]")
ax.set_title("Colour-driving collected efficiency vs standard efficiencies")
ax.legend()
plt.tight_layout()
plt.show()

5. Near fields

|E| scattered‑like

- highlights resonant field localization
- shows Bragg confinement / leakage
- good for understanding colour‑driving resonances

|S| delta

- shows net energy redirection
- matches your forward/backward integrated analysis
- best visual correlate to g, backward fraction, colour strength

Streamlines

- dense backward streamlines → strong reflectance‑like behaviour
- lateral bending → side scattering
- straight-through → forward dominance

In [ ]:
# Choose wavelengths & plane for field inspection
# Wavelengths for near-field inspection
field_wavelengths_nm = [
    float(row_sel["design_peak_wavelength_nm"]),
]

# Optional: add colour-driving peak wavelength if available
if "q_collected_geom_peak_wavelength_nm" in row_sel:
    field_wavelengths_nm.append(
        float(row_sel["q_collected_geom_peak_wavelength_nm"])
    )

# Choose plane and resolution
FIELD_PLANE = "xz"     # "xy", "xz", or "yz"
FIELD_NPTS = 401
EXTENT_FACTOR = 2.5   # extent = factor × outer radius

In [ ]:
# Helper to reconstruct geometry / stack for the selected case
# This is useful because SweepCaseResult stores the scattering result, but near-field calculations often need the stack again.
# This helper assumes your sweep was run in "peak_wavelength" mode and that your notebook still has:

# mat_A
# mat_B
# FIXED_GEOMETRY_KWARGS

# already defined.
from bragg_onion.geometry import build_bragg_onion_from_peak_wavelength, resolve_layer_stack

def rebuild_stack_from_selected_row(
    row_sel,
    *,
    material_a,
    material_b,
    fixed_geometry_kwargs=None,
    extinction_modifiers=None,
    extra_outer_shell=None,
):
    """
    Reconstruct geometry + resolved stack for one selected sweep row.
    """
    fixed_geometry_kwargs = dict(fixed_geometry_kwargs or {})

    geom_kwargs = dict(fixed_geometry_kwargs)
    geom_kwargs.update(
        {
            "peak_wavelength_m": float(row_sel["peak_wavelength_m"]),
            "n_layers": int(row_sel["n_layers"]),
            "outer_layer": str(row_sel["outer_layer"]),
            "core_thickness_factor": float(row_sel["core_thickness_factor"]),
        }
    )

    geometry = build_bragg_onion_from_peak_wavelength(
        material_a=material_a,
        material_b=material_b,
        extra_outer_shell=extra_outer_shell,
        **geom_kwargs,
    )

    stack = resolve_layer_stack(
        geometry=geometry,
        material_a=material_a,
        material_b=material_b,
        extinction_modifiers=extinction_modifiers,
    )

    return geometry, stack

In [ ]:
# Rebuild stack for the selected case
geometry_sel, stack_sel = rebuild_stack_from_selected_row(
    row_sel,
    material_a=mat_A,
    material_b=mat_B,
    fixed_geometry_kwargs=FIXED_GEOMETRY_KWARGS,
)

In [ ]:
# Compute field maps
from bragg_onion.fields import compute_field_map

field_maps = {}

for wl_nm in field_wavelengths_nm:
    print(f"Computing field map at {wl_nm:.1f} nm")

    field_maps[wl_nm] = compute_field_map(
        stack=stack_sel,
        wavelength_m=wl_nm * 1e-9,
        n_medium=medium,
        plane=FIELD_PLANE,
        npts=FIELD_NPTS,
        extent_outer_radius_factor=EXTENT_FACTOR,
    )

print("Field maps computed:", list(field_maps.keys()))

In [ ]:
# field magnitude plots (|E|, |H|, |S|)
# A) Total field magnitude |E|
from bragg_onion.fields import plot_field_magnitude

for wl_nm, fm in field_maps.items():
    plot_field_magnitude(
        fm,
        quantity="E",
        kind="total",
        scale="linear",
        clip_percentile_low=5,
        clip_percentile_high=99.5,
        cmap="viridis",
    )
    plt.show()

# B) Scattered‑like |E| (very informative!)
for wl_nm, fm in field_maps.items():
    plot_field_magnitude(
        fm,
        quantity="E",
        kind="scattered",
        scale="linear",
        clip_percentile_low=0,
        clip_percentile_high=80,
        cmap="magma",
    )
    plt.show()

#  |S| magnitude (total energy flow)
for wl_nm, fm in field_maps.items():
    plot_field_magnitude(
        fm,
        quantity="S",
        kind="total",
        scale="db",
        clip_percentile_low=5,
        clip_percentile_high=99.5,
        cmap="inferno",
    )
    plt.show()

In [ ]:
# Poynting streamlines
# A) Delta flow
from bragg_onion.fields import plot_poynting_streamlines

for wl_nm, fm in field_maps.items():
    plot_poynting_streamlines(
        fm,
        flow_kind="delta",
        background_quantity="S",
        background_kind="delta",
        background_scale="db",
        # background_floor=1e-14,
        # background_cmap="inferno",
        background_clip_percentile_high=99.7,
        streamline_density=1.4,
        streamline_color="white",
        # streamline_linewidth=0.9,
        normalize_vectors=True,
        min_speed_fraction=0,
        mask_inside_sphere=True,
    )
    plt.show()

# B) Total flow (reference)
for wl_nm, fm in field_maps.items():
    plot_poynting_streamlines(
        fm,
        flow_kind="total",
        background_quantity="S",
        background_kind="total",
        background_scale="db",
        background_clip_percentile_high=99.7,
        streamline_density=1.4,
        streamline_color="white",
        normalize_vectors=False,
        min_speed_fraction=0.00,
        mask_inside_sphere=True,
    )
    plt.show()

# C) Scattered‑like flow
for wl_nm, fm in field_maps.items():
    plot_poynting_streamlines(
        fm,
        flow_kind="scattered",
        background_quantity="S",
        background_kind="scattered",
        background_scale="db",
        background_clip_percentile_high=99.7,
        streamline_density=1.4,
        streamline_color="white",
        normalize_vectors=True,
        min_speed_fraction=0.00,
        mask_inside_sphere=True,
    )
    plt.show()

In [ ]:
# Advanced streamline seeding
# If you want controlled streamline injection, e.g. from above the particle
# This is excellent for:
# - visualizing forward vs backward redirection
# - understanding how Bragg layering redirects energy
# - comparing different layer counts at the same wavelength
from bragg_onion.fields import make_line_seeds

# Seed a vertical line above the particle
seed_points_nm = make_line_seeds(
    start_nm=(-500, 800),
    end_nm=(500, 800),
    n_seeds=25,
)

for wl_nm, fm in field_maps.items():
    plot_poynting_streamlines(
        fm,
        flow_kind="delta",
        background_quantity="S",
        background_kind="delta",
        background_scale="db",
        background_clip_percentile_high=99.4,
        streamline_density=1.0,
        streamline_color="cyan",
        # start_points_nm=seed_points_nm,
        normalize_vectors=True,
        min_speed_fraction=1e-100,
        mask_inside_sphere=True,
    )
    plt.show()

Automation of producing plots above by going through top cases
This assumes:
-  store_full_results = True

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

def analyze_case_part3(
    row_sel,
    sweep_result,
    *,
    material_a,
    material_b,
    medium,
    fixed_geometry_kwargs=None,
    alpha_deg=30.0,
    field_plane="xz",
    field_npts=401,
    field_extent_factor=2.5,
    save_dir=None,
    save_pdf=True,
    save_png=False,
    show=True,
):
    """
    Run the Part-3 analysis for one selected sweep case.

    Parameters
    ----------
    row_sel :
        One row from df_plot / sweep dataframe
    sweep_result :
        SweepResult object with case_results stored
    material_a, material_b :
        Materials used to rebuild the stack for field plots
    medium :
        Surrounding medium
    fixed_geometry_kwargs :
        Optional fixed geometry kwargs used in the original sweep
    alpha_deg :
        Cone half-angle for forward/side/backward partition plots
    field_plane :
        Plane for field maps ("xy", "xz", or "yz")
    field_npts :
        Number of grid points per axis for field maps
    field_extent_factor :
        Extent factor relative to outer sphere radius
    save_dir :
        Directory where figures should be saved
    save_pdf :
        If True, save PDF versions
    save_png :
        If True, save PNG versions
    show :
        If True, display figures in the notebook
    """
    case_idx = int(row_sel["case_index"])
    case = next(c for c in sweep_result.case_results if c.case_index == case_idx)

    scat = case.scattering_result
    integ = case.integrated_result

    # --------------------------------------------------------
    # Ensure output directory exists
    # --------------------------------------------------------
    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------
    metrics = scattering_metrics_dataframe(
        scat,
        design_wavelength_nm=row_sel["design_peak_wavelength_nm"],
        alpha_deg=alpha_deg,
    )
    display(metrics)

    # --------------------------------------------------------
    # Differential scattering
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_differential_scattering_vs_angle(
        scat,
        wavelengths_nm=[450, 500, row_sel["design_peak_wavelength_nm"], 600, 650],
        normalization="geom",
        ax=ax,
    )
    ax.set_title("Differential scattering vs angle")
    plt.tight_layout()

    if save_dir is not None:
        if save_pdf:
            fig.savefig(save_dir / f"case_{case_idx:03d}_diff_scattering.pdf")
        if save_png:
            fig.savefig(save_dir / f"case_{case_idx:03d}_diff_scattering.png", dpi=600)

    if show:
        plt.show()
    else:
        plt.close(fig)

    # --------------------------------------------------------
    # Forward / side / backward spectra
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_forward_side_backward_vs_wavelength(
        scat,
        alpha_deg=alpha_deg,
        normalize="geom",
        ax=ax,
    )
    plt.tight_layout()

    if save_dir is not None:
        if save_pdf:
            fig.savefig(save_dir / f"case_{case_idx:03d}_partition_spectra.pdf")
        if save_png:
            fig.savefig(save_dir / f"case_{case_idx:03d}_partition_spectra.png", dpi=600)

    if show:
        plt.show()
    else:
        plt.close(fig)

    # --------------------------------------------------------
    # Pie chart
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_partition_pie_at_wavelength(
        scat,
        alpha_deg=alpha_deg,
        design_wavelength_nm=row_sel["design_peak_wavelength_nm"],
        ax=ax,
    )
    plt.tight_layout()

    if save_dir is not None:
        if save_pdf:
            fig.savefig(save_dir / f"case_{case_idx:03d}_pie.pdf")
        if save_png:
            fig.savefig(save_dir / f"case_{case_idx:03d}_pie.png", dpi=600)

    if show:
        plt.show()
    else:
        plt.close(fig)

    # --------------------------------------------------------
    # Rebuild geometry / stack for field maps
    # --------------------------------------------------------
    geometry, stack = rebuild_stack_from_selected_row(
        row_sel,
        material_a=material_a,
        material_b=material_b,
        fixed_geometry_kwargs=fixed_geometry_kwargs,
    )

    field_map = compute_field_map(
        stack=stack,
        wavelength_m=row_sel["design_peak_wavelength_nm"] * 1e-9,
        n_medium=medium,
        plane=field_plane,
        npts=field_npts,
        extent_outer_radius_factor=field_extent_factor,
    )

    # --------------------------------------------------------
    # Electric field magnitude
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_field_magnitude(
        field_map,
        quantity="E",
        kind="scattered",
        scale="db",
        ax=ax,
    )
    plt.tight_layout()

    if save_dir is not None:
        if save_pdf:
            fig.savefig(save_dir / f"case_{case_idx:03d}_field_E.pdf")
        if save_png:
            fig.savefig(save_dir / f"case_{case_idx:03d}_field_E.png", dpi=600)

    if show:
        plt.show()
    else:
        plt.close(fig)

    # --------------------------------------------------------
    # Delta-flow Poynting streamlines
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_poynting_streamlines(
        field_map,
        flow_kind="delta",
        normalize_vectors=True,
        mask_inside_sphere=True,
        ax=ax,
    )
    plt.tight_layout()

    if save_dir is not None:
        if save_pdf:
            fig.savefig(save_dir / f"case_{case_idx:03d}_poynting_delta.pdf")
        if save_png:
            fig.savefig(save_dir / f"case_{case_idx:03d}_poynting_delta.png", dpi=600)

    if show:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
# Top-N automation
TOP_N = 5
top_cases = df_plot.sort_values(
    ["eta_C", "q_collected_geom_max"],
    ascending=[False, False],
).head(TOP_N)

OUT_DIR = OUTPUT_DIR / "part3_top_cases"

# polar angle for dividing the solid angle into thirds
sphere_area_fract = 1/3
alpha_deg= np.rad2deg(np.arccos(1-4*np.pi * sphere_area_fract / (2*np.pi)))

for _, row in top_cases.iterrows():
    analyze_case_part3(
        row,
        sweep_result,
        material_a=mat_A,
        material_b=mat_B,
        medium=medium,
        fixed_geometry_kwargs=FIXED_GEOMETRY_KWARGS,
        alpha_deg=alpha_deg,
        save_dir=OUT_DIR,
        show=False,
    )

In [ ]:
# Manual case selection
selected = df_plot[
    (df_plot["design_peak_wavelength_nm"] == 550.0) &
    (df_plot["n_layers"] == 20)
]

# polar angle for dividing the solid angle into thirds
sphere_area_fract = 1/3
alpha_deg= np.rad2deg(np.arccos(1-4*np.pi * sphere_area_fract / (2*np.pi)))

for _, row in selected.iterrows():
    analyze_case_part3(
        row,
        sweep_result,
        material_a=mat_A,
        material_b=mat_B,
        medium=medium,
        fixed_geometry_kwargs=FIXED_GEOMETRY_KWARGS,
        alpha_deg=alpha_deg,
        show=True,
    )

## Compare PS/P2VP with another system.
# Workflow
1. Run another sweep
2. Add a label
3. Concatenate
4. Plot comparisions

df_ps_p2vp = make_plotting_aliases(sweep_result_ps_p2vp.dataframe)
df_ps_p2vp["system"] = "PS/P2VP"

df_other = make_plotting_aliases(sweep_result_other.dataframe)
df_other["system"] = "Other"

df_compare = pd.concat([df_ps_p2vp, df_other])

# Then
for system, grp in df_compare.groupby("system"):
    plt.plot(grp["design_peak_wavelength_nm"], grp["eta_C"], label=system)

plt.legend()
plt.xlabel("Design wavelength [nm]")
plt.ylabel("η_C")
plt.show()

# Verify that incident field is correctly computed

In [ ]:
import importlib
import bragg_onion.fields as bo_fields
importlib.reload(bo_fields)

from bragg_onion.fields import compute_field_map

In [ ]:
from bragg_onion.materials import ConstantDispersion

# Same as medium (air)
mat_same = ConstantDispersion.from_nk("same_as_medium", n=1.0, k=0.0)
medium_same = ConstantDispersion.from_nk("air", n=1.0, k=0.0)

In [ ]:
from bragg_onion.geometry import build_bragg_onion_from_peak_wavelength, resolve_layer_stack

geom_kwargs_test = {
    "peak_wavelength_m": float(row_sel["peak_wavelength_m"]),
    "n_layers": int(row_sel["n_layers"]),
    "outer_layer": str(row_sel["outer_layer"]),
    "core_thickness_factor": float(row_sel["core_thickness_factor"]),
}

geometry_test = build_bragg_onion_from_peak_wavelength(
    material_a=mat_same,
    material_b=mat_same,
    **geom_kwargs_test,
)

stack_test = resolve_layer_stack(
    geometry=geometry_test,
    material_a=mat_same,
    material_b=mat_same,
)

In [ ]:
import numpy as np
import pandas as pd

def finite_ratio(numerator, denominator, *, mask=None, floor=1e-300):
    """
    Robust max-ratio diagnostic that ignores non-finite values.
    """
    num = np.asarray(numerator, dtype=float)
    den = np.asarray(denominator, dtype=float)

    valid = np.isfinite(num) & np.isfinite(den)
    if mask is not None:
        valid &= mask

    n_valid = int(np.count_nonzero(valid))
    n_total = int(valid.size)

    if n_valid == 0:
        return {
            "ratio": np.nan,
            "n_valid": n_valid,
            "n_total": n_total,
            "num_max": np.nan,
            "den_max": np.nan,
        }

    num_max = float(np.nanmax(np.abs(num[valid])))
    den_max = float(np.nanmax(np.abs(den[valid])))

    ratio = num_max / max(den_max, floor)

    return {
        "ratio": ratio,
        "n_valid": n_valid,
        "n_total": n_total,
        "num_max": num_max,
        "den_max": den_max,
    }


def diagnose_field_consistency(field_map):
    """
    Diagnose same-index-sphere consistency for:
    - E_sca_like vs E_total
    - H_sca_like vs H_total
    - S_delta vs S_inc
    - S_sca_like vs S_inc

    Reports both:
    - all points
    - outside-sphere only
    """
    outside = field_map.radial_distance_m > field_map.outer_radius_m

    rows = []

    tests = [
        ("E_sca_like / E_total", field_map.e_sca_like_mag, field_map.e_total_mag),
        ("H_sca_like / H_total", field_map.h_sca_like_mag, field_map.h_total_mag),
        ("S_delta / S_inc", field_map.s_delta_mag, field_map.s_inc_mag),
        ("S_sca_like / S_inc", field_map.s_sca_like_mag, field_map.s_inc_mag),
    ]

    for name, num, den in tests:
        all_diag = finite_ratio(num, den, mask=None)
        out_diag = finite_ratio(num, den, mask=outside)

        rows.append({
            "quantity": name,
            "ratio_all": all_diag["ratio"],
            "valid_all": f"{all_diag['n_valid']}/{all_diag['n_total']}",
            "ratio_outside": out_diag["ratio"],
            "valid_outside": f"{out_diag['n_valid']}/{out_diag['n_total']}",
            "num_max_all": all_diag["num_max"],
            "den_max_all": all_diag["den_max"],
        })

    return pd.DataFrame(rows)

In [ ]:
from bragg_onion.fields import compute_field_map

wl_test_nm = float(row_sel["design_peak_wavelength_nm"])

field_test = compute_field_map(
    stack=stack_test,
    wavelength_m=wl_test_nm * 1e-9,
    n_medium=medium_same,
    plane="xz",
    npts=301,
    extent_outer_radius_factor=2.5,
)

print(f"Field test computed at {wl_test_nm:.1f} nm")

In [ ]:
# Cell A — inspect the mean outside Poynting direction
outside = field_test.radial_distance_m > field_test.outer_radius_m

Sz_total = field_test.s_total[..., 2]
Sz_inc = field_test.s_inc[..., 2]

print("Mean outside S_total,z :", np.nanmean(Sz_total[outside]))
print("Mean outside S_inc,z   :", np.nanmean(Sz_inc[outside]))

In [ ]:
# Cell B — compare outside H_total against H_inc component-wise
Htot_y = field_test.h_total[..., 1]
Hinc_y = field_test.h_inc[..., 1]

valid = outside & np.isfinite(Htot_y) & np.isfinite(Hinc_y)

print("Outside max |H_total_y|:", np.nanmax(np.abs(Htot_y[valid])))
print("Outside max |H_inc_y|  :", np.nanmax(np.abs(Hinc_y[valid])))

# Ratio of maxima
print("Outside max-ratio |H_total_y| / |H_inc_y|:",
      np.nanmax(np.abs(Htot_y[valid])) / max(np.nanmax(np.abs(Hinc_y[valid])), 1e-300))

In [ ]:
# Cell C — estimate the best complex scaling factor for H
Htot_y = field_test.h_total[..., 1]
Hinc_y = field_test.h_inc[..., 1]

valid = outside & np.isfinite(Htot_y) & np.isfinite(Hinc_y)

num = np.vdot(Hinc_y[valid], Htot_y[valid])   # <Hinc, Htot>
den = np.vdot(Hinc_y[valid], Hinc_y[valid])   # <Hinc, Hinc>

alpha_H = num / den
print("Best-fit complex scaling alpha_H:", alpha_H)
print("Magnitude:", abs(alpha_H))
print("Phase ", np.angle(alpha_H))

In [ ]:
def max_rel(residual, reference, floor=1e-30):
    ref = np.maximum(np.max(np.asarray(reference, dtype=float)), floor)
    return float(np.max(np.asarray(residual, dtype=float)) / ref)

diag = {
    "max |E_sca_like| / max |E_inc|": max_rel(field_test.e_sca_like_mag, field_test.e_total_mag),
    "max |H_sca_like| / max |H_inc|": max_rel(field_test.h_sca_like_mag, field_test.h_total_mag),
    "max |S_delta| / max |S_inc|": max_rel(field_test.s_delta_mag, field_test.s_inc_mag),
    "max |S_sca_like| / max |S_inc|": max_rel(field_test.s_sca_like_mag, field_test.s_inc_mag),
}

for k, v in diag.items():
    print(f"{k}: {v:.3e}")

In [ ]:
diag_df = diagnose_field_consistency(field_test)
display(diag_df)

In [ ]:
from bragg_onion.fields import plot_field_magnitude

plot_field_magnitude(
    field_test,
    quantity="E",
    kind="scattered",
    scale="linear",
)
plt.title("Test: |E_sca_like| for same-index sphere")
plt.show()

In [ ]:
plot_field_magnitude(
    field_test,
    quantity="S",
    kind="delta",
    scale="linear",
)
plt.title("Test: |S_delta| for same-index sphere")
plt.show()

In [ ]:
from bragg_onion.fields import plot_poynting_streamlines

plot_poynting_streamlines(
    field_test,
    flow_kind="delta",
    background_quantity="S",
    background_kind="delta",
    background_scale="linear",
    normalize_vectors=True,
    min_speed_fraction=0.0,
    mask_inside_sphere=False,
)
plt.title("Test: delta-flow streamlines for same-index sphere")
plt.show()

# Test vector field plots quiver like

In [ ]:
import importlib
import bragg_onion.fields as bo_fields

importlib.reload(bo_fields)

from bragg_onion.fields import (
    compute_field_map,
    plot_field_magnitude,
    plot_poynting_streamlines,
    plot_poynting_vectors,
    make_line_seeds,
)

In [ ]:
wl_test_nm = float(row_sel["design_peak_wavelength_nm"])

field_map = compute_field_map(
    stack=stack_sel,
    wavelength_m=wl_test_nm * 1e-9,
    n_medium=medium,
    plane="xz",
    npts=401,
    extent_outer_radius_factor=4,
)

print(f"Computed field map at {wl_test_nm:.1f} nm")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

plot_poynting_vectors(
    field_map,
    flow_kind="delta",
    background_quantity="S",
    background_kind="delta",
    background_scale="db",
    background_clip_percentile_high=99.7,
    step=14,
    normalize_vectors=True,
    min_speed_fraction=0.01,
    mask_inside_sphere=True,
    vector_color="white",
    pivot="tail",
    ax=ax,
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

plot_poynting_vectors(
    field_map,
    flow_kind="scattered",
    background_quantity="S",
    background_kind="scattered",
    background_scale="db",
    background_clip_percentile_high=99.7,
    step=14,
    normalize_vectors=True,
    min_speed_fraction=0.005,
    mask_inside_sphere=True,
    vector_color="white",
    pivot="mid",
    ax=ax,
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

plot_poynting_vectors(
    field_map,
    flow_kind="total",
    background_quantity="S",
    background_kind="total",
    background_scale="db",
    background_clip_percentile_high=99.7,
    step=14,
    normalize_vectors=True,
    min_speed_fraction=0.01,
    mask_inside_sphere=False,
    vector_color="white",
    pivot="mid",
    ax=ax,
)

plt.show()

# Test seeding on sphere surface


In [ ]:
# Notebook Cell A — reload the module
import importlib
import bragg_onion.fields as bo_fields

importlib.reload(bo_fields)

from bragg_onion.fields import (
    compute_field_map,
    plot_field_magnitude,
    plot_poynting_streamlines,
    plot_poynting_vectors,
    make_line_seeds,
    make_circle_seeds,
)

In [ ]:
# Notebook Cell B — make surface seeds for the current field map
# This uses the outer sphere radius from the FieldMapResult.
# Full circle
surface_seeds_nm = make_circle_seeds(
    radius_nm=field_map.outer_radius_m * 1e9,
    n_seeds=50,
    radial_offset_nm=2.0,   # recommended so seeds lie just outside the masked boundary
)

surface_seeds_nm[:5]

# Upper half only
# If you want only seeds on the upper hemisphere of the plot:
surface_seeds_top_nm = make_circle_seeds(
    radius_nm=field_map.outer_radius_m * 1e9,
    n_seeds=25,
    start_angle_deg=0.0,
    end_angle_deg=180.0,
    radial_offset_nm=2.0,
)

# Back side only
# If you want seeds only on the backward-facing half in the xz plane, for example:
surface_seeds_back_nm = make_circle_seeds(
    radius_nm=field_map.outer_radius_m * 1e9,
    n_seeds=25,
    start_angle_deg=90.0,
    end_angle_deg=270.0,
    radial_offset_nm=2.0,
)

In [ ]:
# Example: streamlines seeded on the sphere surface
# Delta flow (recommended)
fig, ax = plt.subplots(figsize=(6, 5))

plot_poynting_streamlines(
    field_map,
    flow_kind="delta",
    background_quantity="S",
    background_kind="delta",
    background_scale="db",
    background_clip_percentile_high=99.7,
    streamline_density=20.0,
    streamline_color="white",
    start_points_nm=surface_seeds_nm,
    normalize_vectors=True,
    min_speed_fraction=0.01,
    mask_inside_sphere=False,
    ax=ax,
)

plt.show()

In [ ]:
# Scattered-like flow
fig, ax = plt.subplots(figsize=(6, 5))

plot_poynting_streamlines(
    field_map,
    flow_kind="scattered",
    background_quantity="S",
    background_kind="scattered",
    background_scale="db",
    background_clip_percentile_high=99.7,
    streamline_density=80.0,
    streamline_color="white",
    start_points_nm=surface_seeds_nm,
    normalize_vectors=True,
    min_speed_fraction=0.001,
    mask_inside_sphere=False,
    ax=ax,
)

plt.show()

In [ ]:
# Example: vectors instead of streamlines, but still surface-focused
# If you want a simpler test, plot vectors and overlay the seed points:
fig, ax = plt.subplots(figsize=(6, 5))

plot_poynting_vectors(
    field_map,
    flow_kind="delta",
    background_quantity="S",
    background_kind="delta",
    background_scale="db",
    step=14,
    normalize_vectors=True,
    min_speed_fraction=0.01,
    mask_inside_sphere=True,
    vector_color="black",
    ax=ax,
)

ax.scatter(
    surface_seeds_nm[:, 0],
    surface_seeds_nm[:, 1],
    s=15,
    c="red",
    label="surface seeds",
)
ax.legend()

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

# ------------------------------------------------------------
# 1) Background map
# ------------------------------------------------------------
plot_field_magnitude(
    field_map,
    quantity="S",
    kind="scattered",
    scale="db",
    clip_percentile_low=5,
    clip_percentile_high=99.7,
    cmap="magma",
    ax=ax,
)

# ------------------------------------------------------------
# 2) Overlay Poynting vectors
# ------------------------------------------------------------
plot_poynting_vectors(
    field_map,
    flow_kind="scattered",
    background_quantity="none",
    step=14,
    normalize_vectors=True,
    min_speed_fraction=0.01,
    mask_inside_sphere=True,
    vector_color="cyan",
    vector_scale=18,
    vector_width=0.003,
    pivot="mid",
    show_boundaries=False,
    ax=ax,
)

# ------------------------------------------------------------
# 3) Seed points on / just outside sphere surface
# ------------------------------------------------------------
surface_seeds_nm = make_circle_seeds(
    radius_nm=field_map.outer_radius_m * 1e9,
    n_seeds=40,
    radial_offset_nm=2.0,   # try 2.0, 5.0, or 10.0 nm
)

ax.scatter(
    surface_seeds_nm[:, 0],
    surface_seeds_nm[:, 1],
    s=18,
    c="lime",
    edgecolors="black",
    linewidths=0.4,
    label="surface seeds",
    zorder=5,
)

# ------------------------------------------------------------
# 4) Overlay streamlines
# ------------------------------------------------------------
plot_poynting_streamlines(
    field_map,
    flow_kind="scattered",
    background_quantity="none",
    streamline_density=30.0,
    streamline_color="white",
    start_points_nm=surface_seeds_nm,
    normalize_vectors=True,
    min_speed_fraction=0.0001,
    mask_inside_sphere=False,
    show_boundaries=True,
    ax=ax,
)

ax.legend(loc="upper right")
ax.set_title(f"Delta-flow: background + vectors + seeds + streamlines at {field_map.wavelength_m*1e9:.1f} nm")
plt.show()

# Streamlines with improved arrowheads and seeds at incident plane edge for E and S plots

In [ ]:
import importlib
import bragg_onion.fields as bo_fields

importlib.reload(bo_fields)

from bragg_onion.fields import (
    compute_field_map,
    plot_field_magnitude,
    plot_poynting_streamlines,
    plot_poynting_vectors,
    make_line_seeds,
    make_circle_seeds,
    make_edge_seeds,
)

In [ ]:
edge_seeds_nm = make_edge_seeds(
    field_map,
    edge="min_coord2",   # incoming edge for +z propagation in xz / yz
    n_seeds=40,
    margin_nm=5.0,
    start_fraction=0.10,
    end_fraction=0.90,
)

edge_seeds_nm[:5]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

plot_poynting_streamlines(
    field_map,
    flow_kind="total",
    background_quantity="S",
    background_kind="total",
    background_scale="db",
    background_clip_percentile_high=99.7,
    streamline_density=6.0,
    streamline_color="white",
    streamline_linewidth=0.8,
    start_points_nm=edge_seeds_nm,
    normalize_vectors=True,
    min_speed_fraction=0.01,
    mask_inside_sphere=False,
    arrowstyle="simple,head_length=1.0,head_width=0.35,tail_width=0.15",
    arrowsize=1.3,
    integration_direction="forward",
    ax=ax,
)

ax.scatter(
    edge_seeds_nm[:, 0],
    edge_seeds_nm[:, 1],
    s=18,
    c="lime",
    edgecolors="black",
    linewidths=0.4,
    label="edge seeds",
)
ax.legend(loc="upper right")

plt.show()

Si / Ag / Si sphere at 800 nm

εSi​=13.64+0.047i
εAg=−28.05+1.525i\varepsilon_{\mathrm{Ag}} = -28.05 + 1.525iεAg​=−28.05+1.525i
λ=800 nm\lambda = 800 \,\mathrm{nm}λ=800nm

Geometry:

core width = 17.74 nm (Si)
inner shell width = 23.31 nm (Ag)
outer shell width = 22.95 nm (Si)

That is an A–B–A sphere:

A = Si
B = Ag
outer layer = A
3 total layers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from bragg_onion.materials import ConstantDispersion
from bragg_onion.geometry import build_bragg_onion_from_thicknesses, resolve_layer_stack
from bragg_onion.fields import (
    compute_field_map,
    plot_field_magnitude,
    plot_poynting_streamlines,
    make_edge_seeds,
)

# ============================================================
# 1) Material definitions from dielectric functions
# ============================================================

eps_Si = 13.64 + 0.047j
eps_Ag = -28.05 + 1.525j

# Principal square root to obtain n + i k
nk_Si = np.sqrt(eps_Si)
nk_Ag = np.sqrt(eps_Ag)

print("Si: n + i k =", nk_Si)
print("Ag: n + i k =", nk_Ag)

mat_Si = ConstantDispersion.from_nk("Si", n=float(np.real(nk_Si)), k=float(np.imag(nk_Si)))
mat_Ag = ConstantDispersion.from_nk("Ag", n=float(np.real(nk_Ag)), k=float(np.imag(nk_Ag)))

# Surrounding medium = air
medium = ConstantDispersion.from_nk("air", n=1.0, k=0.0)

# ============================================================
# 2) Geometry definition: Si core / Ag shell / Si shell
# ============================================================

WL_NM = 800.0
wavelength_m = WL_NM * 1e-9

core_width_nm = 17.74   # Si core
inner_width_nm = 23.31  # Ag shell
outer_width_nm = 22.95  # Si shell

geometry = build_bragg_onion_from_thicknesses(
    t_a_m=outer_width_nm * 1e-9,                     # A = Si shell thickness
    t_b_m=inner_width_nm * 1e-9,                     # B = Ag shell thickness
    outer_layer="A",
    n_layers=3,                                      # A-B-A
    core_thickness_factor=core_width_nm / outer_width_nm,
)

print("Geometry summary:")
print("  outer radius [nm] =", geometry.outer_radius_m * 1e9)
print("  diameter [nm]     =", geometry.diameter_m * 1e9)

stack = resolve_layer_stack(
    geometry=geometry,
    material_a=mat_Si,
    material_b=mat_Ag,
)

print("Resolved radii [nm]:", stack.radii_m * 1e9)
print("Layer labels:", stack.layer_labels)

# ============================================================
# 3) Compute field map
# ============================================================

field_map = compute_field_map(
    stack=stack,
    wavelength_m=wavelength_m,
    n_medium=medium,
    plane="xz",
    npts=401,
    extent_outer_radius_factor=2.0,
)

print("Computed field map at", WL_NM, "nm")

# ============================================================
# 4) Plot |E| in linear scale
# ============================================================

fig, ax = plt.subplots(figsize=(6.5, 5.5))

plot_field_magnitude(
    field_map,
    quantity="E",
    kind="total",
    scale="linear",
    cmap="jet",
    ax=ax,
)

ax.set_title(f"|E| total, linear scale, λ = {WL_NM:.1f} nm")
plt.tight_layout()
plt.show()

# ============================================================
# 5) Plot total Poynting vector distribution with streamlines
#    (incident side edge seeding)
# ============================================================

fig, ax = plt.subplots(figsize=(6.5, 5.5))

plot_poynting_streamlines(
    field_map,
    flow_kind="total",
    background_quantity="S",
    background_kind="total",
    background_scale="linear",
    start_points_nm=edge_seeds_nm,
    streamline_density=2.0,
    normalize_vectors=True,
    min_speed_fraction=0.0,          # MUST be zero
    mask_inside_sphere=False,        # MUST be false
    integration_direction="forward", # MUST be forward
    minlength=0.01,
    maxlength=0.6,
    broken_streamlines=False,
    arrowsize=1.1,
    arrowstyle="simple,head_length=1.0,head_width=0.35,tail_width=0.15",
    ax=ax,
)

ax.set_title("Total Poynting flow (linear)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))

plot_poynting_streamlines(
    field_map,
    flow_kind="delta",
    background_quantity="S",
    background_kind="delta",
    background_scale="linear",
    streamline_density=6.0,
    streamline_color="white",
    streamline_linewidth=0.8,
    start_points_nm=surface_seeds_nm,
    normalize_vectors=True,
    min_speed_fraction=0.01,
    mask_inside_sphere=False,
    arrowstyle="simple,head_length=1.0,head_width=0.35,tail_width=0.15",
    arrowsize=1,
    integration_direction="forward",
    ax=ax,
)

# ax.scatter(
#     edge_seeds_nm[:, 0],
#     edge_seeds_nm[:, 1],
#     s=16,
#     c="lime",
#     edgecolors="black",
#     linewidths=0.4,
#     label="edge seeds",
# )
ax.scatter(
    surface_seeds_nm[:, 0],
    surface_seeds_nm[:, 1],
    s=18,
    c="lime",
    edgecolors="black",
    linewidths=0.4,
    label="surface seeds",
    zorder=5,
)
ax.legend(loc="upper right")

ax.set_title(f"Delta Poynting flow + streamlines, linear scale, λ = {WL_NM:.1f} nm")
plt.tight_layout()
plt.show()

# Compare two systems

In [ ]:
# Mode A — same design target
geometry_mode="peak_wavelength"
# Mode B — same actual geometry
geometry_mode="thicknesses"

In [ ]:
# Cell 1 — helper to label a sweep DataFram
def labelled_sweep_dataframe(sweep_result, label: str) -> pd.DataFrame:
    """
    Convert a SweepResult into a plotting DataFrame and attach a material-system label.
    """
    df = make_plotting_aliases(sweep_result.dataframe).copy()
    df["material_system"] = label
    return df

In [ ]:
# Cell 2 — helper to compare one metric between systems
def compare_metric_between_systems(
    df_compare: pd.DataFrame,
    *,
    metric: str,
    x: str = "design_peak_wavelength_nm",
    n_layers: int | None = None,
    outer_layer: str | None = None,
    ax=None,
):
    """
    Plot one metric vs x for multiple material systems.
    """
    df = df_compare.copy()

    if n_layers is not None:
        df = df[df["n_layers"] == n_layers]
    if outer_layer is not None:
        df = df[df["outer_layer"] == outer_layer]

    if ax is None:
        fig, ax = plt.subplots(figsize=(6.5, 4.0))

    for system_name, grp in df.groupby("material_system"):
        grp = grp.sort_values(x)
        ax.plot(grp[x], grp[metric], marker="o", linewidth=1.8, label=system_name)

    ax.set_xlabel(x)
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} comparison")
    ax.legend()
    plt.tight_layout()
    return ax

In [ ]:
# Cell 3 — helper to match the “same nominal case” across systems
def get_matching_case_row(
    df_plot: pd.DataFrame,
    *,
    design_peak_wavelength_nm: float,
    n_layers: int,
    outer_layer: str,
):
    """
    Return the first matching row for a given nominal sweep coordinate.
    """
    subset = df_plot[
        np.isclose(df_plot["design_peak_wavelength_nm"], float(design_peak_wavelength_nm), atol=1e-6)
        & (df_plot["n_layers"] == int(n_layers))
        & (df_plot["outer_layer"].astype(str) == str(outer_layer))
    ]

    if subset.empty:
        raise ValueError("No matching case found.")
    return subset.iloc[0]

# Define your second material system

In [ ]:
# Cell 4 — define another system (template)
# Example A — from files
# Example: replace these paths and names with your second system
FILE_PMMA = r"C:\Users\SchwarzN\OneDrive - Université de Fribourg\Institution\P1_BraggSpericalPigments\Simulation\RefractiveIndices\PolymethylMethacrylate.txt"
FILE_PS = r"C:\Users\SchwarzN\OneDrive - Université de Fribourg\Institution\P1_BraggSpericalPigments\Simulation\RefractiveIndices\PS(hPS)\x=0.0.txt"
FILE_P2VP = r"C:\Users\SchwarzN\OneDrive - Université de Fribourg\Institution\P1_BraggSpericalPigments\Simulation\RefractiveIndices\P2VP(PDP)\x=0.0.txt"


material_specs_other = {
    "P2VP": MaterialFileSpec(
        name="P2VP",
        path=FILE_P2VP,
        wavelength_unit="nm",
        skiprows=2,
        names=["Wavelength", "RefractiveIndex", "k"],
        n_column="RefractiveIndex",
        k_column="k",
        extrapolation="extrapolate",
    ),
    "PMMA": MaterialFileSpec(
        name="PMMA",
        path=FILE_PMMA,
        wavelength_unit="um",
        skiprows=9,
        names=["Wavelength", "RefractiveIndex", "k"],
        n_column="RefractiveIndex",
        k_column="k",
        extrapolation="extrapolate",
    ),
}

loaded_other = load_materials(material_specs_other)

mat_A_other = loaded_other["P2VP"]
mat_B_other = loaded_other["PMMA"]

In [ ]:
# # Example B — constant dispersions
# mat_A_other = ConstantDispersion.from_nk("A_other", n=1.80, k=0.00)
# mat_B_other = ConstantDispersion.from_nk("B_other", n=1.45, k=0.00)

# Run the second sweep

In [ ]:
# Cell 5 — run the sweep for the second system
sweep_result_other = run_bragg_onion_sweep(
    geometry_mode="peak_wavelength",
    parameter_grid=PARAMETER_GRID,
    material_a=mat_A_other,
    material_b=mat_B_other,
    n_medium=medium,
    wavelengths_m=wavelengths_m,
    theta_rad=theta_rad,
    collection_na=COLLECTION_NA,
    collection_direction=COLLECTION_DIRECTION,
    integration_quantity_for_colour=INTEGRATION_QUANTITY_FOR_COLOUR,
    colour_wavelength_min_nm=None,
    colour_wavelength_max_nm=None,
    colour_normalize_input=False,
    colour_normalization="max",   # irrelevant when normalize_input=False
    fixed_geometry_kwargs=FIXED_GEOMETRY_KWARGS,
    store_full_results=True,
    progress=True,
)

# Combine the two systems into one comparison DataFrame

In [ ]:
# Cell 6 — merge and label the sweep results
df_ps_p2vp = labelled_sweep_dataframe(sweep_result, "PS/P2VP")
df_other = labelled_sweep_dataframe(sweep_result_other, "Other system")

df_compare = pd.concat([df_ps_p2vp, df_other], ignore_index=True)

display(df_compare.head())

# Make system-comparison plots

In [ ]:
# Cell 7 — compare eta_C
compare_metric_between_systems(
    df_compare,
    metric="eta_C",
    x="design_peak_wavelength_nm",
    n_layers=20,
    outer_layer="A",
)
plt.show()

In [ ]:
# Cell 8 — compare q_collected_geom_max
compare_metric_between_systems(
    df_compare,
    metric="q_collected_geom_max",
    x="design_peak_wavelength_nm",
    n_layers=20,
    outer_layer="A",
)
plt.show()

In [ ]:
# Cell 9 — compare colour coordinates (for example L*)
compare_metric_between_systems(
    df_compare,
    metric="L",
    x="design_peak_wavelength_nm",
    n_layers=20,
    outer_layer="A",
)
plt.show()

# Compare one specific nominal case between systems

Now suppose you want to compare, for example:

design wavelength = 550 nm
n_layers = 20
outer_layer = "A"

in both systems.

In [ ]:
# Cell 10 — select the matching case from both systems
row_ps_p2vp = get_matching_case_row(
    df_ps_p2vp,
    design_peak_wavelength_nm=550.0,
    n_layers=20,
    outer_layer="A",
)

row_other = get_matching_case_row(
    df_other,
    design_peak_wavelength_nm=550.0,
    n_layers=20,
    outer_layer="A",
)

display(pd.DataFrame([row_ps_p2vp, row_other])[[
    "material_system",
    "case_index",
    "design_peak_wavelength_nm",
    "n_layers",
    "outer_layer",
    "eta_C",
    "q_collected_geom_max",
    "hex",
]])

# Compare Part‑3 reports for the matched cases

Now you can run your polished automated report function on both.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
import pandas as pd


from bragg_onion.materials import MaterialFileSpec, ConstantDispersion, load_materials
from bragg_onion.sweep import (
    run_bragg_onion_sweep,
    make_plotting_aliases,
    plot_sweep_metric,
    plot_sweep_heatmap,
    plot_sweep_colour_strip,
)

from bragg_onion.scattering_analysis import (
    plot_differential_scattering_vs_angle,
    plot_polar_scattering,
    plot_forward_side_backward_vs_wavelength,
    plot_partition_pie_at_wavelength,
    print_scattering_metrics,
    scattering_metrics_dataframe,
)

from bragg_onion.colour_solid_plotting import (
    load_rosch_macadam_colour_solid,
    plot_rosch_macadam_colour_solid,
    add_cielab_point,
)

import bragg_onion.fields as bo_fields

from bragg_onion.fields import (
    compute_field_map,
    plot_field_magnitude,
    plot_poynting_streamlines,
    plot_poynting_vectors,
    make_line_seeds,
    make_circle_seeds,
    make_edge_seeds,
)

In [ ]:
# Helper to reconstruct geometry / stack for the selected case
# This is useful because SweepCaseResult stores the scattering result, but near-field calculations often need the stack again.
# This helper assumes your sweep was run in "peak_wavelength" mode and that your notebook still has:

# mat_A
# mat_B
# FIXED_GEOMETRY_KWARGS

# already defined.
from bragg_onion.geometry import build_bragg_onion_from_peak_wavelength, resolve_layer_stack

def rebuild_stack_from_selected_row(
    row_sel,
    *,
    material_a,
    material_b,
    fixed_geometry_kwargs=None,
    extinction_modifiers=None,
    extra_outer_shell=None,
):
    """
    Reconstruct geometry + resolved stack for one selected sweep row.
    """
    fixed_geometry_kwargs = dict(fixed_geometry_kwargs or {})

    geom_kwargs = dict(fixed_geometry_kwargs)
    geom_kwargs.update(
        {
            "peak_wavelength_m": float(row_sel["peak_wavelength_m"]),
            "n_layers": int(row_sel["n_layers"]),
            "outer_layer": str(row_sel["outer_layer"]),
            "core_thickness_factor": float(row_sel["core_thickness_factor"]),
        }
    )

    geometry = build_bragg_onion_from_peak_wavelength(
        material_a=material_a,
        material_b=material_b,
        extra_outer_shell=extra_outer_shell,
        **geom_kwargs,
    )

    stack = resolve_layer_stack(
        geometry=geometry,
        material_a=material_a,
        material_b=material_b,
        extinction_modifiers=extinction_modifiers,
    )

    return geometry, stack

In [ ]:
def set_publication_style():
    plt.rcParams.update({
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "legend.fontsize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "lines.linewidth": 1.6,
        "figure.figsize": (6.0, 4.0),
        "savefig.bbox": "tight",
    })


def save_figure_multi(
    base_path: Path,
    *,
    fig=None,
    save_pdf: bool = True,
    save_png: bool = False,
    dpi: int = 600,
    show: bool = True,
    close: bool = False,
):
    if fig is None:
        fig = plt.gcf()

    base_path = Path(base_path)
    base_path.parent.mkdir(parents=True, exist_ok=True)

    if save_pdf:
        fig.savefig(base_path.with_suffix(".pdf"))
    if save_png:
        fig.savefig(base_path.with_suffix(".png"), dpi=dpi)

    if show:
        plt.show()
    if close:
        plt.close(fig)

In [ ]:
def get_top_n_cases(
    df_plot: pd.DataFrame,
    *,
    n: int = 5,
    sort_by=("eta_C", "q_collected_geom_max"),
    ascending=(False, False),
):
    return df_plot.sort_values(list(sort_by), ascending=list(ascending)).head(n).copy()


def get_cases_by_index(df_plot: pd.DataFrame, case_indices):
    case_indices = list(case_indices)
    return df_plot[df_plot["case_index"].isin(case_indices)].copy()


def get_cases_by_filters(df_plot: pd.DataFrame, filters: list[dict]):
    rows = []
    for filt in filters:
        mask = np.ones(len(df_plot), dtype=bool)

        for key, val in filt.items():
            if key not in df_plot.columns:
                raise KeyError(f"Column '{key}' not found in df_plot.")
            if isinstance(val, float):
                mask &= np.isclose(df_plot[key].to_numpy(dtype=float), val, atol=1e-6)
            else:
                mask &= df_plot[key].astype(str).to_numpy() == str(val)

        subset = df_plot.loc[mask]
        if subset.empty:
            print(f"No match for filter: {filt}")
        else:
            rows.append(subset.iloc[0])

    if not rows:
        raise ValueError("No cases matched the requested filters.")

    return pd.DataFrame(rows)

In [ ]:
def make_case_folder_name(row_sel, *, system_label: str | None = None) -> str:
    parts = []
    if system_label:
        parts.append(system_label.replace(" ", "_").replace("/", "_"))

    parts.extend([
        f"case_{int(row_sel['case_index']):03d}",
        f"lam_{float(row_sel['design_peak_wavelength_nm']):.0f}nm",
        f"layers_{int(row_sel['n_layers'])}",
        f"outer_{str(row_sel['outer_layer'])}",
    ])

    return "_".join(parts)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

def analyze_case_part3(
    row_sel,
    sweep_result,
    *,
    material_a,
    material_b,
    medium,
    fixed_geometry_kwargs=None,
    alpha_deg=30.0,
    field_plane="xz",
    field_npts=401,
    field_extent_factor=2.5,
    save_dir=None,
    save_pdf=True,
    save_png=False,
    show=True,
):
    """
    Run the Part-3 analysis for one selected sweep case.

    Parameters
    ----------
    row_sel :
        One row from df_plot / sweep dataframe
    sweep_result :
        SweepResult object with case_results stored
    material_a, material_b :
        Materials used to rebuild the stack for field plots
    medium :
        Surrounding medium
    fixed_geometry_kwargs :
        Optional fixed geometry kwargs used in the original sweep
    alpha_deg :
        Cone half-angle for forward/side/backward partition plots
    field_plane :
        Plane for field maps ("xy", "xz", or "yz")
    field_npts :
        Number of grid points per axis for field maps
    field_extent_factor :
        Extent factor relative to outer sphere radius
    save_dir :
        Directory where figures should be saved
    save_pdf :
        If True, save PDF versions
    save_png :
        If True, save PNG versions
    show :
        If True, display figures in the notebook
    """
    case_idx = int(row_sel["case_index"])
    case = next(c for c in sweep_result.case_results if c.case_index == case_idx)

    scat = case.scattering_result
    integ = case.integrated_result

    # --------------------------------------------------------
    # Ensure output directory exists
    # --------------------------------------------------------
    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------
    metrics = scattering_metrics_dataframe(
        scat,
        design_wavelength_nm=row_sel["design_peak_wavelength_nm"],
        alpha_deg=alpha_deg,
    )
    display(metrics)

    # --------------------------------------------------------
    # Differential scattering
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_differential_scattering_vs_angle(
        scat,
        wavelengths_nm=[450, 500, row_sel["design_peak_wavelength_nm"], 600, 650],
        normalization="geom",
        ax=ax,
    )
    ax.set_title("Differential scattering vs angle")
    plt.tight_layout()

    if save_dir is not None:
        if save_pdf:
            fig.savefig(save_dir / f"case_{case_idx:03d}_diff_scattering.pdf")
        if save_png:
            fig.savefig(save_dir / f"case_{case_idx:03d}_diff_scattering.png", dpi=600)

    if show:
        plt.show()
    else:
        plt.close(fig)

    # --------------------------------------------------------
    # Forward / side / backward spectra
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_forward_side_backward_vs_wavelength(
        scat,
        alpha_deg=alpha_deg,
        normalize="geom",
        ax=ax,
    )
    plt.tight_layout()

    if save_dir is not None:
        if save_pdf:
            fig.savefig(save_dir / f"case_{case_idx:03d}_partition_spectra.pdf")
        if save_png:
            fig.savefig(save_dir / f"case_{case_idx:03d}_partition_spectra.png", dpi=600)

    if show:
        plt.show()
    else:
        plt.close(fig)

    # --------------------------------------------------------
    # Pie chart
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_partition_pie_at_wavelength(
        scat,
        alpha_deg=alpha_deg,
        design_wavelength_nm=row_sel["design_peak_wavelength_nm"],
        ax=ax,
    )
    plt.tight_layout()

    if save_dir is not None:
        if save_pdf:
            fig.savefig(save_dir / f"case_{case_idx:03d}_pie.pdf")
        if save_png:
            fig.savefig(save_dir / f"case_{case_idx:03d}_pie.png", dpi=600)

    if show:
        plt.show()
    else:
        plt.close(fig)

    # --------------------------------------------------------
    # Rebuild geometry / stack for field maps
    # --------------------------------------------------------
    geometry, stack = rebuild_stack_from_selected_row(
        row_sel,
        material_a=material_a,
        material_b=material_b,
        fixed_geometry_kwargs=fixed_geometry_kwargs,
    )

    field_map = compute_field_map(
        stack=stack,
        wavelength_m=row_sel["design_peak_wavelength_nm"] * 1e-9,
        n_medium=medium,
        plane=field_plane,
        npts=field_npts,
        extent_outer_radius_factor=field_extent_factor,
    )

    # --------------------------------------------------------
    # Electric field magnitude
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_field_magnitude(
        field_map,
        quantity="E",
        kind="scattered",
        scale="db",
        ax=ax,
    )
    plt.tight_layout()

    if save_dir is not None:
        if save_pdf:
            fig.savefig(save_dir / f"case_{case_idx:03d}_field_E.pdf")
        if save_png:
            fig.savefig(save_dir / f"case_{case_idx:03d}_field_E.png", dpi=600)

    if show:
        plt.show()
    else:
        plt.close(fig)

    # --------------------------------------------------------
    # Delta-flow Poynting streamlines
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_poynting_streamlines(
        field_map,
        flow_kind="delta",
        normalize_vectors=True,
        mask_inside_sphere=True,
        ax=ax,
    )
    plt.tight_layout()

    if save_dir is not None:
        if save_pdf:
            fig.savefig(save_dir / f"case_{case_idx:03d}_poynting_delta.pdf")
        if save_png:
            fig.savefig(save_dir / f"case_{case_idx:03d}_poynting_delta.png", dpi=600)

    if show:
        plt.show()
    else:
        plt.close(fig)

In [ ]:


def analyze_case_part3_report(
    row_sel,
    sweep_result,
    *,
    material_a,
    material_b,
    medium,
    fixed_geometry_kwargs=None,
    alpha_deg=30.0,
    field_plane="xz",
    field_npts=401,
    field_extent_factor=2.5,
    save_root=None,
    system_label: str | None = None,
    save_pdf=True,
    save_png=False,
    show=True,
):
    set_publication_style()

    case_idx = int(row_sel["case_index"])
    case = next(c for c in sweep_result.case_results if c.case_index == case_idx)

    scat = case.scattering_result
    integ = case.integrated_result

    # --------------------------------------------------------
    # Build case directory
    # --------------------------------------------------------
    case_dir = None
    if save_root is not None:
        case_dir = Path(save_root) / make_case_folder_name(row_sel, system_label=system_label)
        case_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------
    metrics = scattering_metrics_dataframe(
        scat,
        design_wavelength_nm=row_sel["design_peak_wavelength_nm"],
        alpha_deg=alpha_deg,
    )
    display(metrics)

    if case_dir is not None:
        metrics.to_csv(case_dir / "metrics.csv", index=False)

    # --------------------------------------------------------
    # Differential scattering
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_differential_scattering_vs_angle(
        scat,
        wavelengths_nm=[450, 500, row_sel["design_peak_wavelength_nm"], 600, 650],
        normalization="geom",
        ax=ax,
    )
    ax.set_title("Differential scattering vs angle")
    plt.tight_layout()

    if case_dir is not None:
        save_figure_multi(case_dir / "diff_scattering", fig=fig, save_pdf=save_pdf, save_png=save_png, show=show, close=False)
    elif show:
        plt.show()

    # --------------------------------------------------------
    # Forward / side / backward spectra
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_forward_side_backward_vs_wavelength(
        scat,
        alpha_deg=alpha_deg,
        normalize="geom",
        ax=ax,
    )
    ax.set_title(f"Forward / side / backward efficiencies (α = {alpha_deg:.1f}°)")
    plt.tight_layout()

    if case_dir is not None:
        save_figure_multi(case_dir / "partition_spectra", fig=fig, save_pdf=save_pdf, save_png=save_png, show=show, close=False)
    elif show:
        plt.show()

    # --------------------------------------------------------
    # Pie chart
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_partition_pie_at_wavelength(
        scat,
        alpha_deg=alpha_deg,
        design_wavelength_nm=row_sel["design_peak_wavelength_nm"],
        ax=ax,
    )
    plt.tight_layout()

    if case_dir is not None:
        save_figure_multi(case_dir / "partition_pie", fig=fig, save_pdf=save_pdf, save_png=save_png, show=show, close=False)
    elif show:
        plt.show()

    # --------------------------------------------------------
    # Rebuild geometry / stack for field maps
    # --------------------------------------------------------
    geometry, stack = rebuild_stack_from_selected_row(
        row_sel,
        material_a=material_a,
        material_b=material_b,
        fixed_geometry_kwargs=fixed_geometry_kwargs,
    )

    field_map = compute_field_map(
        stack=stack,
        wavelength_m=row_sel["design_peak_wavelength_nm"] * 1e-9,
        n_medium=medium,
        plane=field_plane,
        npts=field_npts,
        extent_outer_radius_factor=field_extent_factor,
    )

    # --------------------------------------------------------
    # E-field magnitude
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_field_magnitude(
        field_map,
        quantity="E",
        kind="scattered",
        scale="db",
        ax=ax,
    )
    plt.tight_layout()

    if case_dir is not None:
        save_figure_multi(case_dir / "field_E_scattered", fig=fig, save_pdf=save_pdf, save_png=save_png, show=show, close=False)
    elif show:
        plt.show()

    # --------------------------------------------------------
    # Delta-flow streamlines (edge-seeded, longer)
    # --------------------------------------------------------
    edge_seeds_nm = make_edge_seeds(
        field_map,
        edge="min_coord2",
        n_seeds=35,
        margin_nm=5.0,
        start_fraction=0.10,
        end_fraction=0.90,
    )

    fig, ax = plt.subplots()
    plot_poynting_streamlines(
        field_map,
        flow_kind="delta",
        background_quantity="S",
        background_kind="delta",
        background_scale="db",
        background_clip_percentile_high=99.7,
        streamline_density=4.0,
        streamline_color="white",
        streamline_linewidth=0.8,
        start_points_nm=edge_seeds_nm,
        normalize_vectors=True,
        min_speed_fraction=0.001,
        mask_inside_sphere=False,
        integration_direction="forward",
        minlength=0.05,
        maxlength=20.0,
        broken_streamlines=False,
        arrowstyle="simple,head_length=1.0,head_width=0.35,tail_width=0.15",
        arrowsize=1.3,
        ax=ax,
    )
    plt.tight_layout()

    if case_dir is not None:
        save_figure_multi(case_dir / "poynting_delta_streamlines", fig=fig, save_pdf=save_pdf, save_png=save_png, show=show, close=False)
    elif show:
        plt.show()

    # --------------------------------------------------------
    # Delta-flow vectors
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_poynting_vectors(
        field_map,
        flow_kind="delta",
        background_quantity="S",
        background_kind="delta",
        background_scale="db",
        background_clip_percentile_high=99.7,
        step=12,
        normalize_vectors=True,
        min_speed_fraction=0.001,
        mask_inside_sphere=False,
        vector_color="white",
        vector_scale=18,
        vector_width=0.003,
        pivot="mid",
        quiver_headwidth=2.6,
        quiver_headlength=4.8,
        quiver_headaxislength=4.8,
        ax=ax,
    )
    plt.tight_layout()

    if case_dir is not None:
        save_figure_multi(case_dir / "poynting_delta_vectors", fig=fig, save_pdf=save_pdf, save_png=save_png, show=show, close=False)
    elif show:
        plt.show()

    return metrics

In [ ]:
# Cell 11 — generate polished case reports for both systems
COMPARE_REPORT_ROOT = OUTPUT_DIR / "material_system_comparison_reports"

analyze_case_part3_report(
    row_ps_p2vp,
    sweep_result,
    material_a=mat_A,
    material_b=mat_B,
    medium=medium,
    fixed_geometry_kwargs=FIXED_GEOMETRY_KWARGS,
    alpha_deg=alpha_deg,
    save_root=COMPARE_REPORT_ROOT,
    system_label="PS_P2VP",
    save_pdf=True,
    save_png=False,
    show=True,
)

analyze_case_part3_report(
    row_other,
    sweep_result_other,
    material_a=mat_A_other,
    material_b=mat_B_other,
    medium=medium,
    fixed_geometry_kwargs=FIXED_GEOMETRY_KWARGS,
    alpha_deg=alpha_deg,
    save_root=COMPARE_REPORT_ROOT,
    system_label="Other_system",
    save_pdf=True,
    save_png=False,
    show=True,
)

# Optional: compare one metric for all layer counts

If you want a broad overview:

In [ ]:
# Cell 12 — compare eta_C for all layer counts
fig, ax = plt.subplots(figsize=(7, 4.5))

for (system_name, n_layers), grp in df_compare.groupby(["material_system", "n_layers"]):
    grp = grp.sort_values("design_peak_wavelength_nm")
    ax.plot(
        grp["design_peak_wavelength_nm"],
        grp["eta_C"],
        marker="o",
        linewidth=1.5,
        label=f"{system_name}, n={n_layers}",
    )

ax.set_xlabel("Design peak wavelength [nm]")
ax.set_ylabel("eta_C")
ax.set_title("eta_C comparison across systems and layer counts")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

# Resonance analysis for one case

In [ ]:
# Cell 1 — extract basic spectra
scat = case_sel.scattering_result
integ = case_sel.integrated_result

wl_nm = scat.wavelengths_m * 1e9

Qsca = scat.qsca
Qext = scat.qext
g = scat.g

q_back = integ.q_collected_geom

In [ ]:
# Cell 2 — plot far‑field resonance indicators
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(wl_nm, Qsca, label="Qsca", linewidth=1.8)
ax.plot(wl_nm, q_back / np.max(q_back) * np.max(Qsca),
        label="q_collected_geom (scaled)", linestyle="--", linewidth=1.6)

ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("Scattering strength [arb.]")
ax.set_title("Far-field resonance indicators")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 3 — asymmetry parameter (very important)
# Interpretation
# sharp dips / sign changes in g(λ) = redistribution resonance
# strong backward scattering often coincides with g → 0 or negative excursions

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(wl_nm, g, linewidth=1.8)
ax.axhline(0.0, color="gray", linestyle="--")

ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("g")
ax.set_title("Asymmetry parameter (directionality resonance)")
plt.tight_layout()
plt.show()


# 4. Near‑field resonance diagnostic (key insight)

Instead of plotting a single field map, do this:

> Track the maximum near‑field enhancement as a function of wavelength

This is the most reliable near‑field resonance indicator.

In [ ]:
# Cell 4 — compute max |E| vs wavelength
from bragg_onion.fields import compute_field_map

def max_field_enhancement_vs_wavelength(
    stack,
    wavelengths_nm,
    *,
    n_medium,
    plane="xz",
    npts=301,
    extent_factor=1.8,
):
    max_E = []

    for wl_nm in wavelengths_nm:
        fm = compute_field_map(
            stack=stack,
            wavelength_m=wl_nm * 1e-9,
            n_medium=n_medium,
            plane=plane,
            npts=npts,
            extent_outer_radius_factor=extent_factor,
        )
        max_E.append(np.nanmax(fm.e_total_mag))

    return np.asarray(max_E)

In [ ]:
# Cell 5 — run it for your selected case
geometry, stack = rebuild_stack_from_selected_row(
    row_sel,
    material_a=mat_A,
    material_b=mat_B,
    fixed_geometry_kwargs=FIXED_GEOMETRY_KWARGS,
)

# Use a reduced wavelength grid for speed
wl_probe_nm = wl_nm[::5]   # every 5 nm
max_E = max_field_enhancement_vs_wavelength(
    stack,
    wl_probe_nm,
    n_medium=medium,
)

In [ ]:
# Cell 6 — plot near‑field resonance
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(wl_probe_nm, max_E / np.max(max_E), linewidth=1.8)
ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("max(|E|) (normalized)")
ax.set_title("Near-field resonance indicator")
plt.tight_layout()
plt.show()

In [ ]:
# 5. Combine all resonance indicators in one plot (very powerful)
fig, ax = plt.subplots(figsize=(7.5, 4.5))

ax.plot(wl_nm, Qsca / np.max(Qsca), label="Qsca", linewidth=1.8)
ax.plot(wl_nm, q_back / np.max(q_back), label="q_collected_geom", linewidth=1.8)
ax.plot(wl_probe_nm, max_E / np.max(max_E), "o-", label="max |E|", linewidth=1.6)

ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("Normalized magnitude")
ax.set_title("Resonance comparison: far-field vs near-field")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 6. Selecting the resonance wavelength for deeper analysis
res_idx = np.argmax(q_back)
wl_res_nm = wl_nm[res_idx]

print(f"Selected resonance wavelength ≈ {wl_res_nm:.1f} nm")

# Analyze MIE Resonances

In [ ]:
# Cell 1 — imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bragg_onion.solver import build_scattnlay_inputs_single_wavelength

try:
    from scattnlay import scattcoeffs
except ImportError as exc:
    raise ImportError(
        "The installed scattnlay package does not expose scattcoeffs. "
        "This modal analysis requires scattcoeffs(x, m)."
    ) from exc

In [ ]:
# Cell 2 — rebuild the selected stack 
geometry_sel, stack_sel = rebuild_stack_from_selected_row(
    row_sel,
    material_a=mat_A,
    material_b=mat_B,
    fixed_geometry_kwargs=FIXED_GEOMETRY_KWARGS,
)

In [ ]:
# Cell 3 — helper: compute coefficients at one wavelength
def mie_coefficients_single_wavelength(
    stack,
    *,
    wavelength_m: float,
    n_medium,
    nmax: int = -1,
    pl: int = -1,
    mp: bool = False,
):
    """
    Compute scattnlay Mie coefficients a_l, b_l for one wavelength.

    Returns
    -------
    dict with:
        terms
        l
        a
        b
        x_outer
        qsca_l
        qsca_a_l
        qsca_b_l
        qext_l
        qabs_l
    """
    x, m, radii_m, n_medium_eval, k_medium = build_scattnlay_inputs_single_wavelength(
        stack=stack,
        wavelength_m=wavelength_m,
        n_medium=n_medium,
    )

    terms, a, b = scattcoeffs(x, m, nmax=nmax, pl=pl, mp=mp)

    a = np.asarray(a, dtype=np.complex128)
    b = np.asarray(b, dtype=np.complex128)
    l = np.arange(1, len(a) + 1, dtype=int)

    # Use the outermost size parameter for the standard external-efficiency formulas
    x_outer = float(np.asarray(x, dtype=float)[-1])

    pref = 2.0 / (x_outer**2)
    factor = pref * (2 * l + 1)

    qsca_a_l = factor * np.abs(a)**2
    qsca_b_l = factor * np.abs(b)**2
    qsca_l = qsca_a_l + qsca_b_l
    qext_l = factor * np.real(a + b)
    qabs_l = qext_l - qsca_l

    return {
        "terms": int(terms),
        "l": l,
        "a": a,
        "b": b,
        "x_outer": x_outer,
        "qsca_l": qsca_l,
        "qsca_a_l": qsca_a_l,
        "qsca_b_l": qsca_b_l,
        "qext_l": qext_l,
        "qabs_l": qabs_l,
    }

In [ ]:
# Cell 4 — choose a resonance wavelength and compute coefficients
# You can use the resonance wavelength from automatic selection above, or use the design wavelength.
# Example: use the q_collected_geom peak if available, otherwise design wavelength
if "q_collected_geom_peak_wavelength_nm" in row_sel:
    wl_res_nm = float(row_sel["q_collected_geom_peak_wavelength_nm"])
else:
    wl_res_nm = float(row_sel["design_peak_wavelength_nm"])

coeff_res = mie_coefficients_single_wavelength(
    stack_sel,
    wavelength_m=wl_res_nm * 1e-9,
    n_medium=medium,
)

print(f"Modal analysis at λ = {wl_res_nm:.1f} nm")
print("Number of terms:", coeff_res["terms"])

In [ ]:
# Cell 5 — tabulate and rank the modal contributions at that wavelength
# This table answers:

# which ℓ dominates
# whether the dominant contribution is electric (a_l) or magnetic (b_l)
# whether the mode is mostly scattering or extinction-dominated
df_modes_res = pd.DataFrame({
    "l": coeff_res["l"],
    "qsca_l": coeff_res["qsca_l"],
    "qsca_a_l": coeff_res["qsca_a_l"],
    "qsca_b_l": coeff_res["qsca_b_l"],
    "qext_l": coeff_res["qext_l"],
    "qabs_l": coeff_res["qabs_l"],
    "|a_l|": np.abs(coeff_res["a"]),
    "|b_l|": np.abs(coeff_res["b"]),
})

df_modes_res_sorted = df_modes_res.sort_values("qsca_l", ascending=False).reset_index(drop=True)
display(df_modes_res_sorted.head(12))

In [ ]:
# Cell 6 — bar chart of modal contributions at the selected wavelength
# Interpretation

# If one or two bars dominate strongly → clean modal resonance
# If many l contribute comparably → broad / geometrically complicated response
# If a_l dominates → electric-like resonance
# If b_l dominates → magnetic-like resonance

TOP_L = 10
df_top = df_modes_res_sorted.head(TOP_L).sort_values("l")

fig, ax = plt.subplots(figsize=(7, 4.5))
xpos = np.arange(len(df_top))

ax.bar(xpos - 0.18, df_top["qsca_a_l"], width=0.36, label="electric (a_l)")
ax.bar(xpos + 0.18, df_top["qsca_b_l"], width=0.36, label="magnetic (b_l)")

ax.set_xticks(xpos)
ax.set_xticklabels([f"l={l}" for l in df_top["l"]])
ax.set_ylabel("Partial scattering efficiency contribution")
ax.set_title(f"Top modal scattering contributions at λ = {wl_res_nm:.1f} nm")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 — helper: compute mode-resolved spectra over wavelength
def mie_coefficient_spectrum(
    stack,
    *,
    wavelengths_nm,
    n_medium,
    nmax: int = -1,
    pl: int = -1,
    mp: bool = False,
):
    """
    Compute coefficient-derived modal spectra over wavelength.

    Returns
    -------
    dict with arrays:
        wavelengths_nm
        qsca_l_spectrum   shape (n_wavelengths, l_max)
        qsca_a_spectrum   shape (n_wavelengths, l_max)
        qsca_b_spectrum   shape (n_wavelengths, l_max)
        qext_l_spectrum   shape (n_wavelengths, l_max)
        terms_per_wavelength
    """
    wavelengths_nm = np.asarray(wavelengths_nm, dtype=float)

    all_coeffs = []
    terms_list = []

    for wl_nm in wavelengths_nm:
        coeff = mie_coefficients_single_wavelength(
            stack,
            wavelength_m=wl_nm * 1e-9,
            n_medium=n_medium,
            nmax=nmax,
            pl=pl,
            mp=mp,
        )
        all_coeffs.append(coeff)
        terms_list.append(coeff["terms"])

    l_max = max(len(c["l"]) for c in all_coeffs)
    n_wl = len(wavelengths_nm)

    qsca_l_spec = np.full((n_wl, l_max), np.nan, dtype=float)
    qsca_a_spec = np.full((n_wl, l_max), np.nan, dtype=float)
    qsca_b_spec = np.full((n_wl, l_max), np.nan, dtype=float)
    qext_l_spec = np.full((n_wl, l_max), np.nan, dtype=float)

    for i, coeff in enumerate(all_coeffs):
        n_terms = len(coeff["l"])
        qsca_l_spec[i, :n_terms] = coeff["qsca_l"]
        qsca_a_spec[i, :n_terms] = coeff["qsca_a_l"]
        qsca_b_spec[i, :n_terms] = coeff["qsca_b_l"]
        qext_l_spec[i, :n_terms] = coeff["qext_l"]

    return {
        "wavelengths_nm": wavelengths_nm,
        "qsca_l_spectrum": qsca_l_spec,
        "qsca_a_spectrum": qsca_a_spec,
        "qsca_b_spectrum": qsca_b_spec,
        "qext_l_spectrum": qext_l_spec,
        "terms_per_wavelength": np.asarray(terms_list, dtype=int),
    }

In [ ]:
# Cell 8 — compute the modal spectra
# For speed, I recommend starting with a coarser wavelength grid.
wl_modal_nm = wl_nm[::5]   # every 5 nm; refine later if needed

modal_spec = mie_coefficient_spectrum(
    stack_sel,
    wavelengths_nm=wl_modal_nm,
    n_medium=medium,
)

In [ ]:
# Cell 9 — rank the modes by their maximum spectral contribution
qsca_l_spec = modal_spec["qsca_l_spectrum"]

mode_max = np.nanmax(qsca_l_spec, axis=0)
l_orders = np.arange(1, qsca_l_spec.shape[1] + 1)

df_mode_rank = pd.DataFrame({
    "l": l_orders,
    "max_qsca_l": mode_max,
}).sort_values("max_qsca_l", ascending=False).reset_index(drop=True)

display(df_mode_rank.head(15))

In [ ]:
# Cell 10 — plot the top N modal spectra
# Interpretation
# This is one of the most useful plots in the whole workflow:

# each curve shows when that multipole turns on resonantly
# the top modes usually line up with the resonances you found in Point 5

TOP_N_MODES = 6
top_l_orders = df_mode_rank.head(TOP_N_MODES)["l"].to_list()

fig, ax = plt.subplots(figsize=(7.5, 4.8))

for l_sel in top_l_orders:
    idx = l_sel - 1
    ax.plot(
        wl_modal_nm,
        qsca_l_spec[:, idx],
        linewidth=1.8,
        label=f"l = {l_sel}",
    )

ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("Partial scattering efficiency contribution")
ax.set_title("Top modal scattering contributions vs wavelength")
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11 — split the top modes into electric vs magnetic contributions
# This helps you identify whether a resonance is:

# electric-dipole-like
# magnetic-quadrupole-like
# etc.

fig, axes = plt.subplots(2, 1, figsize=(7.5, 7.0), sharex=True)

for l_sel in top_l_orders:
    idx = l_sel - 1
    axes[0].plot(
        wl_modal_nm,
        modal_spec["qsca_a_spectrum"][:, idx],
        linewidth=1.6,
        label=f"l = {l_sel}",
    )
    axes[1].plot(
        wl_modal_nm,
        modal_spec["qsca_b_spectrum"][:, idx],
        linewidth=1.6,
        label=f"l = {l_sel}",
    )

axes[0].set_ylabel("Electric partial contribution")
axes[0].set_title("Electric modal contributions (a_l)")
axes[0].legend(ncol=2)

axes[1].set_xlabel("Wavelength [nm]")
axes[1].set_ylabel("Magnetic partial contribution")
axes[1].set_title("Magnetic modal contributions (b_l)")
axes[1].legend(ncol=2)

plt.tight_layout()
plt.show()